# __3- Connecting The Camera__ 

The pipline :

→ Open Camera

→ Read frame

→ Prepare the frame

→ Pass the frame to the model

→ Make preduction

→ Write the predicted gusture on the screen


## 3.1 Import CV Libriry and Prepare Model For Preduction

In [ ]:
!pip install opencv-python

In [42]:

import torch # pytorch main libirary
import torch.nn as nn 
import cv2  
from torchvision import transforms, models
from PIL import Image

# Load Pretrained ResNet-18 Model
model = models.resnet18(pretrained=True)

# Modify the Final Fully Connected Layer where we have 2 classes 
num_classes = 2
model.fc = nn.Linear(model.fc.in_features, num_classes)

# The trained model 
model.load_state_dict(torch.load("best_model.pth", map_location='cpu'))
model.eval()

class_names = ["Open", "Close"]
 
# will used to prepare the frame before send it to model 
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Same transforms used on the dataset
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

C:\Users\rahaf\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\rahaf\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


## 3.2 Open Camera and Start Immediate Preduction

In [45]:
import torch.nn.functional as F
from PIL import Image


img = Image.open(r"D:\Downloadf\gestures_resized\gestures\03_fist\WIN_20260217_19_15_01_Pro.jpg").convert("RGB")

img = transform(img)
img = img.unsqueeze(0)

with torch.no_grad():
    outputs = model(img)
    probs = F.softmax(outputs, dim=1)
    confidence, predicted = torch.max(probs, 1)

print("Prediction:", class_names[predicted.item()])
print("Confidence:", confidence.item())


Prediction: Close
Confidence: 0.5755681991577148


In [ ]:
cap = cv2.VideoCapture(0) # use OpenCV to open the camera 0 means the camera is in the same device

while True: # endless loop to read vedio continuously
    ret, frame = cap.read() # ret is bool value (does the frame token sucssfully?), frame is the capture itself
    
    if not ret: # if capturing the frame faild stop the loop
        break
    
    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    frame = frame[:, w//3 : 3*w//3]  # نقص المنتصف

    # مهم جدًا: نحول BGR → RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # نحول إلى PIL
    img = Image.fromarray(rgb).convert("RGB")

    img = transform(img)
    img = img.unsqueeze(0)
    # transform the colors from BGR to RBG becuse OpenCV reads images in BGR
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    image = Image.fromarray(frame_rgb) # the transform needs the image in PIL

    input_tensor = transform(image).unsqueeze(0) # preprocessing the frame

    # because this is a test phase the gradients calculations should stop to speed up the process
    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1) # converts the output to probabilities , softmax converts values ​​into probabilities between 0 and 1
        confidence, predicted = torch.max(probabilities,1) #choose highest probability

    gesture_name = class_names[predicted.item()] # change class num to name (0 -> fist , 1 -> palm)
    conf_value = confidence.item() # extracting the value of trust

    # write gesture name on the vedio
    cv2.putText(frame,
            f"{gesture_name} ({conf_value:.2f})", # the text that will be shown
            (20, 40), # the place
            cv2.FONT_HERSHEY_SIMPLEX, # text type
            1, # text size
            (0, 255, 0), # color
            2) # text thickness

    cv2.imshow("Hand Gesture Recognition", frame) # show the vedio
    

    
    if cv2.waitKey(1) & 0xFF == ord('q'): # to break the loop
        break

cap.release() # stop camera 
cv2.destroyAllWindows() # close all windows



unsqueeze(0) adds a new dimension so that the shape becomes: 

[1, 3, 224, 224]

Because the model accepts Batch even if it's just one image.

مشاكل الى الان:
- دائما يتوقع يد مفتوحة  

- يظهر توقع حتى لو مافي يد في الكاميرا
